# MoE Playground (HomeMoE)

Модель с Mixture of Experts: `HomeMoEConfig`, `HomeMoEForCausalLM`. Сравнение с плотной LLaMA-style по параметрам и loss.

In [ ]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

from homellm.models.home_model import HomeConfig, HomeForCausalLM
from homellm.models.home_model_moe import HomeMoEConfig, HomeMoEForCausalLM
from homellm.training.pretrain import StreamingTextDataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_PATH = '/app/datasets/fineweb-2_train.jsonl'
ensure_pretrain_dataset(DATA_PATH)  # скачает с HF, если файла нет
SEQ_LEN = 512
BATCH_SIZE = 2
MAX_STEPS = 100

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})

dataset = StreamingTextDataset(DATA_PATH, tokenizer, seq_len=SEQ_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=DataCollatorForLanguageModeling(tokenizer, mlm=False), num_workers=0)

cfg_dense = HomeConfig(vocab_size=len(tokenizer), hidden_size=256, num_hidden_layers=4, num_attention_heads=4, max_position_embeddings=SEQ_LEN)
cfg_moe = HomeMoEConfig(vocab_size=len(tokenizer), hidden_size=256, num_hidden_layers=4, num_attention_heads=4, max_position_embeddings=SEQ_LEN, num_experts=4, num_experts_per_tok=2)

dense = HomeForCausalLM(cfg_dense).to(DEVICE)
moe = HomeMoEForCausalLM(cfg_moe).to(DEVICE)
moe.resize_token_embeddings(len(tokenizer))
print('Dense params:', sum(p.numel() for p in dense.parameters()))
print('MoE params:', sum(p.numel() for p in moe.parameters()))

In [ ]:
opt = torch.optim.AdamW(moe.parameters(), lr=3e-4, weight_decay=0.1)
moe.train()
pbar = tqdm(total=MAX_STEPS, desc='MoE pretrain')
step = 0
for batch in loader:
    out = moe(
        input_ids=batch['input_ids'].to(DEVICE),
        attention_mask=batch.get('attention_mask'),
        labels=batch['labels'].to(DEVICE),
    )
    out.loss.backward()
    torch.nn.utils.clip_grad_norm_(moe.parameters(), 1.0)
    opt.step()
    opt.zero_grad(set_to_none=True)
    pbar.set_postfix({'loss': f'{out.loss.item():.4f}'})
    pbar.update(1)
    step += 1
    if step >= MAX_STEPS:
        break
pbar.close()
print('Done. Save: moe.save_pretrained("/app/out/moe_playground")')